# Где платят больше

Второй вопрос: как предложение по зарплате зависит от города.

Зарплаты приведены к сумме «на руки» (зарплата «до вычета» делится на 1,13),
а «от» — это нижняя граница вилки, а не сама зарплата. Оба ограничения
подписаны на графике.

In [ ]:
import sys
from pathlib import Path


def find_root() -> Path:
    """Корень репозитория: ищем pyproject.toml вверх по дереву."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return here


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from vacancy_dashboard import analysis, clean, collect  # noqa: E402

RAW = ROOT / "data" / "raw" / "vacancies.json"
FIXTURE = ROOT / "tests" / "fixtures" / "vacancies_sample.json"

source = RAW if RAW.exists() else FIXTURE
print(f"Источник данных: {source}")
if source == FIXTURE:
    print(
        "ВНИМАНИЕ: это синтетическая фикстура, а не рыночные данные. "
        "Числа отсюда нельзя публиковать."
    )

frame = clean.drop_archived(clean.build_frame(collect.load(source)))
print(f"Вакансий в выборке: {len(frame)}")
frame.head()

In [ ]:
result, figure = analysis.question_geography(frame)
print(result.text)
result.table

In [ ]:
figure

## Сколько вакансий без зарплаты

Их нельзя выбрасывать молча: без зарплаты — не значит «ноль».

In [ ]:
usable = clean.salary_frame(frame)
print(f"С зарплатой: {len(usable)} из {len(frame)}")
print(f"Без зарплаты: {len(frame) - len(usable)}")